**1: Preparação do Ambiente e Clonagem do Repositório**

In [ ]:
# Célula 1: Instalação de dependências e clonagem do GitHub
!pip -q install transformers pillow torch torchvision

import os

# Clona o repositório diretamente para o Colab
!git clone https://github.com/Thiagofariasl/atelie-generativo-estilo-pixel-art.git

# Define o caminho
dataset_path = "/content/atelie-generativo-estilo-pixel-art/dataset/images"
output_file = os.path.join(dataset_path, "legendas.txt")

print(f"Repositório clonado! Pasta do dataset: {dataset_path}")

**2: Carregamento do Modelo BLIP**

In [ ]:
# Célula 2: Inicialização do BLIP
from transformers import BlipProcessor, BlipForConditionalGeneration
import torch

# Verifica se a GPU (CUDA) está disponível
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Usando processamento em: {device.upper()}")

# Carrega o processador e o modelo
print("Carregando o modelo BLIP (isso pode levar alguns instantes)...")
proc = BlipProcessor.from_pretrained("Salesforce/blip-image-captioning-base")
blip = BlipForConditionalGeneration.from_pretrained("Salesforce/blip-image-captioning-base").to(device)
print("Modelo carregado com sucesso!")

**3: Processamento em Lote (Batch) e Geração do "legendas.txt"**

In [ ]:
# Célula 3: Geração das legendas
from PIL import Image

# DEFINA TOKEN DE ESTILO AQUI (Trigger Word)
# Ele deve ser único para ensinar ao LoRA que essas características pertencem a essa palavra
trigger_word = "estilo_pixel_art,"

# Lista todas as imagens na pasta (suporta jpg e png)
imagens_validas = ('.jpg', '.jpeg', '.png')
lista_imagens = [f for f in os.listdir(dataset_path) if f.lower().endswith(imagens_validas)]

if len(lista_imagens) == 0:
    print("Nenhuma imagem encontrada na pasta dataset. Verifique se elas já foram atualizadas no GitHub.")
else:
    print(f"Iniciando o processamento de {len(lista_imagens)} imagens...\n")

    # Abre o arquivo legendas.txt em modo de escrita
    with open(output_file, "w", encoding="utf-8") as f:
        for img_name in lista_imagens:
            img_path = os.path.join(dataset_path, img_name)

            # Carrega a imagem
            img = Image.open(img_path).convert("RGB")

            # Gera a legenda bruta com o BLIP
            saida = blip.generate(**proc(img, return_tensors="pt").to(device), max_new_tokens=40)
            legenda_bruta = proc.decode(saida[0], skip_special_tokens=True)

            # Concatena o Token de Estilo com a legenda bruta
            legenda_final = f"{trigger_word} {legenda_bruta}"

            # Escreve no arquivo no formato padrão (NomeDaImagem: Legenda)
            # Obs: Alguns scripts de LoRA preferem "NomeDaImagem.txt" separados.
            linha = f"{img_name}: {legenda_final}\n"
            f.write(linha)

            print(f"✅ {img_name} -> {legenda_final}")

    print(f"\n🎉 Arquivo 'legendas.txt' gerado com sucesso em: {output_file}")

**4: Download Seguro para Revisão Humana**

In [ ]:
# Célula 4: Download do arquivo
from google.colab import files

# Baixa o arquivo para a sua máquina
files.download(output_file)
print("Download iniciado. Revise as legendas manualmente antes de subir para o GitHub!")